- 전처리 후 저장 꼭 하기

In [16]:
import re
import pandas as pd

def split_menu_blocks(text_path):
    with open(text_path, "r", encoding="utf-8") as f:
        text = f.read()

    # 메뉴 경로 기준으로 블록 분리
    blocks = re.findall(r"(‘.+?메뉴는.*?)(?=\n‘|\Z)",\\ text, re.DOTALL)

    # DataFrame 생성
    df = pd.DataFrame(blocks, columns=["menu_description_block"])
    df.to_csv('./datasets/menu_description.csv',index=False)
    return df


In [2]:
# import re
# import pandas as pd

# def split_menu_blocks(text_path):
#     with open(text_path, "r", encoding="utf-8") as f:
#         text = f.read()

#     # 개선된 정규표현식
#     blocks = re.findall(r"(‘.+?메뉴.*?)(?=‘|\Z)", text, re.DOTALL)

#     df = pd.DataFrame(blocks, columns=["menu_description_block"])
#     df.to_csv('./datasets/menu_description.csv', index=False)
#     return df


In [17]:
menu_description = split_menu_blocks('./datasets/Menu_description.txt')
menu_description.head(1)

,menu_description_block
0,"‘국내통계 → 한국무역 → 수출입 총괄 → 총괄’ 메뉴는 연도별, 분기별, 월별 단..."


In [18]:
menu_description['menu_description_block'][38]

'‘해외무역통계 → 아시아 → 대만 → 국가별’ 메뉴는 특정 연도 기준으로 대만과 각 국가 간의 수출입 실적을 비교할 수 있는 메뉴이다. 국가별로 수출금액, 수입금액, 수출입 증감률, 무역수지가 제공되며, 정렬 기준에 따라 주요 교역국과의 무역 흐름을 한눈에 파악할 수 있다. \n'

In [19]:
menu_description["menu_path"] = menu_description["menu_description_block"].str.extract(r"(‘.+?’)")[0].str.strip("‘’")
menu_description.head(1)

,menu_description_block,menu_path
0,"‘국내통계 → 한국무역 → 수출입 총괄 → 총괄’ 메뉴는 연도별, 분기별, 월별 단...",국내통계 → 한국무역 → 수출입 총괄 → 총괄


In [20]:
# 컬럼 위치 재정렬 
menu_description = menu_description[["menu_path", "menu_description_block"]]
menu_description.head(1)

,menu_path,menu_description_block
0,국내통계 → 한국무역 → 수출입 총괄 → 총괄,"‘국내통계 → 한국무역 → 수출입 총괄 → 총괄’ 메뉴는 연도별, 분기별, 월별 단..."


# QA 파일

In [21]:
with open('./datasets/Menu_QA.txt', "r", encoding="utf-8") as f:
    text = f.read()

# \n\n 또는 \r\n\r\n 기준으로 블록 나누기
blocks = [block.strip() for block in text.strip().split("\n\n") if block.strip()]

# DataFrame 생성
menu_qa = pd.DataFrame(blocks, columns=["menu_qa"])

menu_qa.head(1)
menu_qa['menu_qa'][0]
# menu_qa

'질문: 한국 전체 수출입 규모나 종합적인 무역 현황 통계를 보고 싶어. 국가별, 품목별 구분 없이 전체 수출입 데이터가 필요해.\n답변: 아래 메뉴에서 해당 데이터를 확인할 수 있습니다:\n▶ 국내통계 → 한국무역 → 수출입 총괄 → 총괄\n🔗 https://stat.kita.net/stat/kts/sum/SumImpExpTotalList.screen'

In [22]:
# menu_qa["menu_path"] = menu_qa["menu_qa"].str.extract(r"▶ (.+?)\n").strip()
# menu_qa["menu_path"] = menu_qa["menu_qa"].str.extract(r"▶ (.+?)\n")[0].strip()
menu_qa["menu_path"] = menu_qa["menu_qa"].str.extract(r"▶ (.+?)\n")[0].str.strip()
menu_qa = menu_qa[["menu_path", "menu_qa"]]
menu_qa.head(1)

,menu_path,menu_qa
0,국내통계 → 한국무역 → 수출입 총괄 → 총괄,"질문: 한국 전체 수출입 규모나 종합적인 무역 현황 통계를 보고 싶어. 국가별, 품..."


# 경로 기준 데이터 프레임 합치기(1)

In [127]:
df = pd.merge(menu_description, menu_qa, on="menu_path", how="outer")

In [128]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239 entries, 0 to 238
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   menu_path               239 non-null    object
 1   menu_description_block  172 non-null    object
 2   menu_qa                 184 non-null    object
dtypes: object(3)
memory usage: 5.7+ KB


In [46]:
missing_rows = df[df["menu_description_block"].isna() | df["menu_qa"].isna()]
missing_rows.head() 

,menu_path,menu_description_block,menu_qa
43,IMF 세계통계 → 세계무역 → - → 국가간 수출입,NaN,질문: 국가 간 수출입 데이터를 비교하고 싶어.\n답변: 아래 메뉴에서 해당 데이터...
44,IMF 세계통계 → 세계무역 → - → 국가별 수출입,NaN,"질문: 세계무역과 교역이 많은 국가들이 어딘지, 국가별 수출입 현황이 어떻게 되는지..."
45,IMF 세계통계 → 세계무역 → - → 국가의 지역별 수출입,NaN,질문: 세계 각국의 수출입 현황 통계를 확인하고 싶어.\n답변: 아래 메뉴에서 해당...
46,IMF 세계통계 → 세계무역 → - → 수출입 매트릭스,NaN,질문: 세계 각국의 수출입 흐름을 매트릭스 형태로 보고 싶어.\n답변: 아래 메뉴에...
47,IMF 세계통계 → 세계무역 → - → 한국과 경쟁국 수출입,NaN,질문: 한국과 경쟁 관계에 있는 주요 국가들의 수출입 현황을 비교하고 싶은데 관련 ...


# 경로 기준 데이터 프레임 합치기(2)

In [61]:
# IMF 세계통계 → 세계무역 → - → 국가간 수출입 에서 -> -  부분 제거 
menu_qa["menu_path"] = menu_qa["menu_path"].str.replace(r"→\s*-\s*→", "→", regex=True)

In [62]:
df = pd.merge(menu_description, menu_qa, on="menu_path", how="outer")
# df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 356 entries, 0 to 355
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   menu_path               356 non-null    object
 1   menu_description_block  172 non-null    object
 2   menu_qa                 184 non-null    object
dtypes: object(3)
memory usage: 8.5+ KB


In [63]:
# missing_rows = df[df["menu_description_block"].isna() | df["menu_qa"].isna()]
missing_rows = df[df["menu_description_block"].isna()]
missing_rows.head() 

,menu_path,menu_description_block,menu_qa
47,IMF 세계통계무역통계로 보는 주요국EUEU무역 총괄,NaN,질문: EU의 전체 수출입 현황이나 무역 현황 통계를 보고 싶어.\n답변: 아래 메...
48,IMF 세계통계무역통계로 보는 주요국EUEU의 10대 수입국,NaN,질문: EU의 주요 수입국 현황을 보고 싶어.\n답변: 아래 메뉴에서 해당 데이터를...
49,IMF 세계통계무역통계로 보는 주요국EUEU의 10대 수입상품,NaN,질문: EU의 주요 수입상품 현황을 보고 싶어.\n답변: 아래 메뉴에서 해당 데이터...
50,IMF 세계통계무역통계로 보는 주요국EUEU의 10대 수출국,NaN,질문: EU의 주요 수출국 현황을 보고 싶어.\n답변: 아래 메뉴에서 해당 데이터를...
51,IMF 세계통계무역통계로 보는 주요국EUEU의 10대 수출상품,NaN,질문: EU의 주요 수출상품 현황을 보고 싶어.\n답변: 아래 메뉴에서 해당 데이터...


# 경로 기준 데이터 프레임 합치기(3)

In [23]:
# 1단계: 중간에 있는 → - → 제거
menu_qa["menu_path"] = menu_qa["menu_path"].str.replace(r"→\s*-\s*→", "→", regex=True)

# 2단계: 마지막에 남은 → - 제거
menu_qa["menu_path"] = menu_qa["menu_path"].str.replace(r"→\s*-\s*$", "", regex=True)


In [38]:
# df = pd.merge(menu_description, menu_qa, on="menu_path", how="outer")
# # df.head()
# df.info()

In [39]:
# missing_rows = df[df["menu_description_block"].isna() | df["menu_qa"].isna()]
# # missing_rows = df[df["menu_description_block"].isna()]
# missing_rows

# 병합 기준 컬럼 문자, 공백 전처리 
- 공백 제거
- 메뉴에서는 -> 메뉴는 통일 (TXT 파일)
- 자사통계 → 수출입 실적 (국가별/ 총괄/ 품목별) 수정

In [24]:
# 양쪽 공백 제거 + 특수문자 제거 + 표준화
for df in [menu_description, menu_qa]:
    df["menu_path"] = df["menu_path"].str.strip()                           # 앞뒤 공백 제거
    df["menu_path"] = df["menu_path"].str.replace(r"\s+", " ", regex=True) # 이중 공백 제거
    df["menu_path"] = df["menu_path"].str.replace("\u200b", "")             # zero-width space 제거


In [25]:
menu_description_qa = pd.merge(menu_description, menu_qa, on="menu_path", how="outer")
# df.head()
menu_description_qa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   menu_path               184 non-null    object
 1   menu_description_block  184 non-null    object
 2   menu_qa                 184 non-null    object
dtypes: object(3)
memory usage: 4.4+ KB


In [26]:
missing_rows = menu_description_qa[menu_description_qa["menu_description_block"].isna() | menu_description_qa["menu_qa"].isna()]
# missing_rows = df[df["menu_description_block"].isna()]
missing_rows

,menu_path,menu_description_block,menu_qa


# 저정

In [28]:
menu_description_qa.to_csv('./datasets/menu_description_qa.csv',index=False)

# 특정 내용 찾기

In [27]:
# df[df["컬럼명"].str.contains("찾을_내용", na=False)]
menu_description[menu_description['menu_description_block'].str.contains("말레이시아", na=False)] # na=False	결측값(NaN)은 무시하고 False로 간주

,menu_path,menu_description_block
33,해외무역통계 → 아시아 → ASEAN → 총괄,‘해외무역통계 → 아시아 → ASEAN → 총괄’ 메뉴는 아세안(ASEAN) 10개...
34,해외무역통계 → 아시아 → ASEAN → 품목별,‘해외무역통계 → 아시아 → ASEAN → 품목별’ 메뉴는 한국과 ASEAN 10개...
35,해외무역통계 → 아시아 → ASEAN → 국가별,‘해외무역통계 → 아시아 → ASEAN → 국가별’ 메뉴는 ASEAN 10개국 (아...


In [235]:
menu_description['menu_description_block'][38]

'‘해외무역통계 → 아시아 → 대만 → 국가별’ 메뉴는 특정 연도 기준으로 대만과 각 국가 간의 수출입 실적을 비교할 수 있는 메뉴이다. 국가별로 수출금액, 수입금액, 수출입 증감률, 무역수지가 제공되며, 정렬 기준에 따라 주요 교역국과의 무역 흐름을 한눈에 파악할 수 있다. \n'

# 그래프는 -> 그래프 구성은

In [191]:
def replace_graph_phrase(file_path, output_path):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    # "그래프는" → "그래프 구성은"
    updated_text = text.replace("그래프는", "그래프 구성은")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(updated_text)

    print(f"✅ 치환 완료: 저장 위치 → {output_path}")


In [ ]:
replace_graph_phrase('./datasets/')